0. Libraries and data

In [ ]:
! pip install -q openneuro-py mne specparam==2.0.0rc7

In [ ]:
import os, openneuro, shutil, glob, mne, specparam

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.spatial.distance import cdist
from scipy.sparse import csr_matrix
from scipy import stats
from specparam import SpectralGroupModel


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
drive_spectra = '/content/drive/MyDrive/Meditation/aperiodic/spectra'
os.makedirs(drive_spectra, exist_ok = True)

drive_epochs = '/content/drive/MyDrive/Meditation/epochs'
os.makedirs(drive_epochs, exist_ok = True)

drive_aperiodic = '/content/drive/MyDrive/Meditation/aperiodic'
os.makedirs(drive_aperiodic, exist_ok = True)

In [ ]:
# Only once

root = '/content/ds001787'

openneuro.download(dataset='ds001787', tag = '1.1.1', target_dir = root,
                   include = ['participants.tsv'])

shutil.copy('/content/ds001787/participants.tsv',
            '/content/drive/MyDrive/Meditation/participants.tsv')

In [ ]:
participants = pd.read_csv(f"/content/drive/MyDrive/Meditation/participants.tsv", sep='\t')
participants['subject'] = participants['participant_id'].str.replace('sub-', '')

In [ ]:
# sub-013/014: EEG matches subject 14's behavioral logs (identity unconfirmed)
excluded_subjects = ['013', '014']

# sub-007 ses-01: comb-like spectral artefact (regular dips ~1.9 Hz apart, 1–128 Hz)
excluded_sessions = [('007', '01')]

1. Bridge check

In [ ]:
def computing_spectra(key, drive_epochs):
  sub, ses = key
  epochs = mne.read_epochs(f"{drive_epochs}/sub-{sub}_ses-{ses}_epo.fif")

  psd = epochs.compute_psd(method = 'welch', picks = 'eeg', fmin = 1, fmax = 60,
                           n_fft = 512, n_overlap = 256, window = 'hann')

  data = psd.get_data()
  frequencies = psd.freqs
  channels = psd.ch_names
  condition = epochs.metadata['condition'].values

  return data, frequencies, channels, condition

In [ ]:
files = sorted(glob.glob(f"{drive_epochs}/sub-*_ses-*_epo.fif"))

for f in files:
  parts = os.path.basename(f).split('_')
  sub = parts[0].replace('sub-','')
  ses = parts[1].replace('ses-','')
  if sub in excluded_subjects or (sub, ses) in excluded_sessions:
    continue

  data, frequencies, channels, condition = computing_spectra((sub, ses), drive_epochs)
  np.savez(f"{drive_spectra}/sub-{sub}_ses-{ses}_psd.npz",
           psd = data, freqs = frequencies, condition = condition, ch_names = channels)


In [ ]:
def bridge_check(key, drive_spectra):
  sub, ses = key
  data = np.load(f"{drive_spectra}/sub-{sub}_ses-{ses}_psd.npz", allow_pickle = True)

  psd = data['psd']
  frequencies = data['freqs']
  condition = data['condition']
  channels = data['ch_names']

  theta_idx = (frequencies >= 4) & (frequencies <= 7)
  alpha_idx = (frequencies >= 9) & (frequencies <= 11)

  theta = 10 * np.log10(psd[:, :, theta_idx].mean(axis = -1) * 1e12)
  alpha = 10 * np.log10(psd[:, :, alpha_idx].mean(axis = -1) * 1e12)

  n_epochs, n_channels = theta.shape
  power_table = pd.DataFrame({
      'epoch': np.repeat(np.arange(n_epochs), n_channels),
      'channel': np.tile(channels, n_epochs),
      'theta': theta.ravel(),
      'alpha': alpha.ravel(),
    })
  power_table['condition'] = np.repeat(condition, n_channels)
  power_table['subject'], power_table['session'] = sub, ses

  return power_table


In [ ]:
files_spectra = sorted(glob.glob(f"{drive_spectra}/sub-*_ses-*_psd.npz"))
power_tables = []

for f in files_spectra:
  parts = os.path.basename(f).split('_')
  sub = parts[0].replace('sub-','')
  ses = parts[1].replace('ses-','')
  if sub in excluded_subjects or (sub, ses) in excluded_sessions:
    continue

  power_tables.append(bridge_check((sub, ses), drive_spectra))

all_power = pd.concat(power_tables, ignore_index = True)
all_power.to_csv(f"{drive_aperiodic}/band_power_welch.csv", index = False)

In [ ]:
# Comparison with welch (not wavelets)

all_power = pd.read_csv(f"{drive_aperiodic}/band_power_welch.csv", dtype = {'subject': str, 'session': str})
n_trials = all_power.groupby(['subject', 'condition']).size().unstack(fill_value = 0) / 64
excluded = n_trials[(n_trials == 0).any(axis = 1)].index.tolist()
print('Excluded:', excluded)

subject_means = (all_power[~all_power['subject'].isin(excluded)]
                 .groupby(['subject', 'condition', 'channel'])[['theta', 'alpha']]
                 .mean().reset_index())

subject_means = subject_means.merge(participants[['subject', 'group']], on = 'subject')

ch_names = mne.channels.make_standard_montage("biosemi64").ch_names

def difference_table(subject_means, group, band):
    data = subject_means[subject_means["group"] == group]
    wide = data.pivot_table(index="subject", columns=["condition", "channel"], values=band)
    return wide["meditation"][ch_names] - wide["mind_wandering"][ch_names]

X_theta_exp    = difference_table(subject_means, "expert", "theta")
X_alpha_exp    = difference_table(subject_means, "expert", "alpha")
X_theta_novice = difference_table(subject_means, "novice", "theta")
X_alpha_novice = difference_table(subject_means, "novice", "alpha")

pos = mne.channels.make_standard_montage("biosemi64").get_positions()["ch_pos"]
coords = np.array([pos[ch] for ch in ch_names])
dist = cdist(coords, coords)
adjacency = csr_matrix(dist < 0.047)

def compare_conditions (x, adjacency, ch_names):

  threshold = stats.t.ppf(0.975, df=len(x) - 1)

  T_obs, clusters, cluster_pv, H0 = mne.stats.permutation_cluster_1samp_test(
    x.values, adjacency = adjacency, threshold = threshold,
    n_permutations = 2048, tail = 0, rng = 123)

  cluster_channels = [[ch_names[i] for i in c[0]]
                      for c, p in zip(clusters, cluster_pv) if p < 0.05]

  t_vals, p_vals = stats.ttest_1samp(x.values, 0)
  reject, p_fdr = mne.stats.fdr_correction(p_vals, alpha=0.05, method="negcorr")
  fdr_channels = [ch for ch, r in zip(ch_names, reject) if r]

  return {"T_obs": T_obs, "cluster_pv": cluster_pv,
          "cluster_channels": cluster_channels, "fdr_channels": fdr_channels}

res_theta_exp = compare_conditions(X_theta_exp, adjacency, ch_names)
res_alpha_exp = compare_conditions(X_alpha_exp, adjacency, ch_names)
res_theta_novice = compare_conditions(X_theta_novice, adjacency, ch_names)
res_alpha_novice = compare_conditions(X_alpha_novice, adjacency, ch_names)

print(res_theta_exp['cluster_pv'], res_theta_exp['cluster_channels'], res_theta_exp['fdr_channels'])
print(res_alpha_exp['cluster_pv'], res_alpha_exp['cluster_channels'], res_alpha_exp['fdr_channels'])
print(res_theta_novice['cluster_pv'], res_theta_novice['cluster_channels'], res_theta_novice['fdr_channels'])
print(res_alpha_novice['cluster_pv'], res_alpha_novice['cluster_channels'], res_alpha_novice['fdr_channels'])


In [ ]:
print('t at FCz, experts:', round(dict(zip(ch_names, res_theta_exp['T_obs']))['FCz'], 2),
      round(dict(zip(ch_names, res_alpha_exp['T_obs']))['FCz'], 2))

3. Specparam and periodic component

In [ ]:
collected = {}

files_spectra = sorted(glob.glob(f"{drive_spectra}/sub-*_ses-*_psd.npz"))

for f in files_spectra:
  parts = os.path.basename(f).split('_')
  sub = parts[0].replace('sub-','')
  ses = parts[1].replace('ses-','')
  if sub in excluded_subjects or (sub, ses) in excluded_sessions:
    continue

  data = np.load(f, allow_pickle = True)
  psd = data['psd']
  condition = data['condition']
  frequencies = data['freqs']
  channels = data['ch_names']

  for e in range(len(condition)):
    key = (sub, condition[e])
    if key not in collected:
      collected[key] = []
    collected[key].append(psd[e])

mean_spectra = {key: np.mean(spectra, axis = 0) for key, spectra in collected.items()}


In [ ]:
spec_log = []

fg = SpectralGroupModel(peak_width_limits = [1, 6], max_n_peaks = 6, min_peak_height = 0.05,
                        peak_threshold = 1.5, aperiodic_mode = 'knee', verbose = False)

fit_idx = (frequencies >= 3) & (frequencies <= 40)

for key, spectrum in mean_spectra.items():
  sub, cond = key
  fg.fit(frequencies, spectrum, freq_range = [3, 40])

  offset = fg.get_params('aperiodic', 'offset')
  knee = fg.get_params('aperiodic', 'knee')
  exponent = fg.get_params('aperiodic', 'exponent')
  freqs_fit = fg.data.freqs

  r_squared = fg.get_metrics('gof')
  error = fg.get_metrics('error')

  log_spectrum = np.log10(mean_spectra[key][:, fit_idx])
  background = offset[:, None] - np.log10(knee[:, None] + freqs_fit[None, :] ** exponent[:, None])
  flat = log_spectrum - background

  theta_idx = (freqs_fit >= 4) & (freqs_fit <= 7)
  alpha_idx = (freqs_fit >= 9) & (freqs_fit <= 11)

  spec_log.append(pd.DataFrame({
      'subject': sub,
      'condition': cond,
      'channel': channels,
      'theta': 10 * flat[:, theta_idx].mean(axis = 1),
      'alpha': 10 * flat[:, alpha_idx].mean(axis = 1),
      'offset': offset,
      'knee': knee,
      'exponent': exponent,
      'knee_freq': knee ** (1 / exponent),
      'r_squared': r_squared,
      'error': error,
    }))

periodic = pd.concat(spec_log, ignore_index = True)
periodic.to_csv(f"{drive_aperiodic}/periodic_specparam.csv", index = False)

In [ ]:
print(len(periodic))
print(periodic.groupby('subject')['alpha'].mean().round(2).head())

In [ ]:
neg = periodic[periodic['knee'] < 0]
print(len(neg), "of", len(periodic))
print(neg.groupby(['subject', 'condition']).size())

In [ ]:
print(periodic[['theta', 'alpha']].isna().sum())     # should be 0 0
print(neg[['r_squared', 'error']].describe().round(3))

In [ ]:
print(periodic[periodic['theta'].isna()][['subject', 'condition', 'channel', 'knee', 'exponent', 'r_squared']])

In [ ]:
for band in ['theta', 'alpha']:
  periodic[band] = periodic.groupby(['subject', 'condition'])[band].transform(lambda x: x.fillna(x.mean()))

print(periodic[['theta', 'alpha']].isna().sum())

In [ ]:
r2_mean, r2_sd = periodic['r_squared'].mean(), periodic['r_squared'].std()
err_mean, err_sd = periodic['error'].mean(), periodic['error'].std()

r2_dist  = abs(periodic['r_squared'] - r2_mean) / r2_sd
err_dist = abs(periodic['error'] - err_mean) / err_sd

periodic['qc_flag'] = (r2_dist > 2.5) | (err_dist > 2.5)

print(periodic['qc_flag'].sum(), "of", len(periodic), "fits flagged")
print(periodic[periodic['qc_flag']].groupby(['subject', 'condition']).size())

In [ ]:
periodic.to_csv(f"{drive_aperiodic}/periodic_specparam.csv", index = False)


In [ ]:
# Comparison with aperiodic

periodic = pd.read_csv(f"{drive_aperiodic}/periodic_specparam.csv", dtype = {'subject': str, 'session': str})

subject_means = (periodic[~periodic['subject'].isin(['017', '018'])]
                 [['subject', 'condition', 'channel', 'theta', 'alpha']]
                 .merge(participants[['subject', 'group']], on = 'subject'))

ch_names = mne.channels.make_standard_montage("biosemi64").ch_names

def difference_table(subject_means, group, band):
    data = subject_means[subject_means["group"] == group]
    wide = data.pivot_table(index="subject", columns=["condition", "channel"], values=band)
    return wide["meditation"][ch_names] - wide["mind_wandering"][ch_names]

X_theta_exp    = difference_table(subject_means, "expert", "theta")
X_alpha_exp    = difference_table(subject_means, "expert", "alpha")
X_theta_novice = difference_table(subject_means, "novice", "theta")
X_alpha_novice = difference_table(subject_means, "novice", "alpha")

pos = mne.channels.make_standard_montage("biosemi64").get_positions()["ch_pos"]
coords = np.array([pos[ch] for ch in ch_names])
dist = cdist(coords, coords)
adjacency = csr_matrix(dist < 0.047)

def compare_conditions (x, adjacency, ch_names):

  threshold = stats.t.ppf(0.975, df=len(x) - 1)

  T_obs, clusters, cluster_pv, H0 = mne.stats.permutation_cluster_1samp_test(
    x.values, adjacency = adjacency, threshold = threshold,
    n_permutations = 2048, tail = 0, rng = 123)

  cluster_channels = [[ch_names[i] for i in c[0]]
                      for c, p in zip(clusters, cluster_pv) if p < 0.05]

  t_vals, p_vals = stats.ttest_1samp(x.values, 0)
  reject, p_fdr = mne.stats.fdr_correction(p_vals, alpha=0.05, method="negcorr")
  fdr_channels = [ch for ch, r in zip(ch_names, reject) if r]

  return {"T_obs": T_obs, "cluster_pv": cluster_pv,
          "cluster_channels": cluster_channels, "fdr_channels": fdr_channels}

res_theta_exp = compare_conditions(X_theta_exp, adjacency, ch_names)
res_alpha_exp = compare_conditions(X_alpha_exp, adjacency, ch_names)
res_theta_novice = compare_conditions(X_theta_novice, adjacency, ch_names)
res_alpha_novice = compare_conditions(X_alpha_novice, adjacency, ch_names)

print(res_theta_exp['cluster_pv'], res_theta_exp['cluster_channels'], res_theta_exp['fdr_channels'], res_theta_exp['T_obs'])
print(res_alpha_exp['cluster_pv'], res_alpha_exp['cluster_channels'], res_alpha_exp['fdr_channels'], res_alpha_exp['T_obs'])
print(res_theta_novice['cluster_pv'], res_theta_novice['cluster_channels'], res_theta_novice['fdr_channels'], res_theta_novice['T_obs'])
print(res_alpha_novice['cluster_pv'], res_alpha_novice['cluster_channels'], res_alpha_novice['fdr_channels'], res_alpha_novice['T_obs'])



In [ ]:
# Sensitivity analysis without epochs < 3

subject_means_min_epochs = subject_means[~subject_means['subject'].isin(['007', '009'])]

for group in ['expert', 'novice']:
  for band in ['theta', 'alpha']:
    X = difference_table(subject_means_min_epochs, group, band)
    res = compare_conditions(X, adjacency, ch_names)
    T = res['T_obs']
    print(group, band, '| n =', len(X), '| cluster p:', np.round(res['cluster_pv'], 3),
          '| FDR:', res['fdr_channels'], '| t_max:', round(T.max(), 2),
          '| t_FCz:', round(dict(zip(ch_names, T))['FCz'], 2))

In [ ]:
aper = (periodic[~periodic['subject'].isin(['017', '018'])]
        [['subject', 'condition', 'channel', 'offset', 'exponent', 'knee_freq']]
        .merge(participants[['subject', 'group']], on = 'subject'))

whole_head = aper.groupby(['subject', 'group', 'condition'])[['offset', 'exponent', 'knee_freq']].median().reset_index()
fcz = aper[aper['channel'] == 'FCz']

def aperiodic_difference(data, param):
  wide = data.pivot_table(index = ['subject', 'group'], columns = 'condition', values = param)
  return (wide['meditation'] - wide['mind_wandering']).rename(param)

for name, data in [('whole head', whole_head), ('FCz', fcz)]:
  diffs = pd.concat([aperiodic_difference(data, p) for p in ['offset', 'exponent', 'knee_freq']], axis = 1)
  print(name)
  print(diffs.groupby('group').agg(['mean', 'std', 'count']).round(3))

In [ ]:
odd = periodic[periodic['knee_freq'] > 40]
print(len(odd), "fits with knee frequency above 40 Hz")
print(odd.groupby(['subject', 'condition']).size())

In [ ]:
theta_freqs = freqs_fit[(freqs_fit >= 4) & (freqs_fit <= 7)]

periodic['theta_background'] = 10 * np.mean(
    periodic['offset'].values[:, None]
    - np.log10(periodic['knee'].values[:, None] + theta_freqs[None, :] ** periodic['exponent'].values[:, None]),
    axis = 1)

periodic['theta_total'] = periodic['theta'] + periodic['theta_background']

data = periodic[(periodic['channel'] == 'FCz') & (~periodic['subject'].isin(['017', '018']))] \
         .merge(participants[['subject', 'group']], on = 'subject')

for part in ['theta_total', 'theta', 'theta_background']:
  wide = data.pivot_table(index = ['subject', 'group'], columns = 'condition', values = part)
  diff = (wide['meditation'] - wide['mind_wandering']).groupby('group')
  print(part)
  print(pd.DataFrame({'mean_dB': diff.mean(), 'sd': diff.std(), 'd_z': diff.mean() / diff.std()}).round(2))


In [ ]:
bands = {'theta': (4, 7), 'alpha': (9, 11)}
peak_log = []
theta_cfs = []

for key, spectrum in mean_spectra.items():
  sub, cond = key
  fg.fit(frequencies, spectrum, freq_range = [3, 40])

  peaks = fg.get_params('periodic')
  peaks = peaks[~np.isnan(peaks).any(axis = 1)]

  table = pd.DataFrame({'subject': sub, 'condition': cond, 'channel': ch_names})
  for band, (low, high) in bands.items():
    in_band = peaks[(peaks[:, 0] >= low) & (peaks[:, 0] <= high)]
    channels_with_peak = np.unique(in_band[:, 3].astype(int))
    table[f'{band}_peak'] = np.isin(np.arange(64), channels_with_peak)

  theta_cfs.extend(peaks[(peaks[:, 0] >= 4) & (peaks[:, 0] <= 7), 0])

  peak_log.append(table)

peak_table = pd.concat(peak_log, ignore_index = True)

In [ ]:
peaks_tested = (peak_table[~peak_table['subject'].isin(['017', '018'])]
                .merge(participants[['subject', 'group']], on = 'subject'))

plt.hist(theta_cfs, bins = np.arange(4, 7.5, 0.5)); plt.xlabel("Theta peak centre (Hz)"); plt.show()

print((peaks_tested.groupby(['group', 'condition'])[['theta_peak', 'alpha_peak']].mean() * 100).round(1))
print((peaks_tested[peaks_tested['channel'] == 'FCz']
       .groupby(['group', 'condition'])[['theta_peak', 'alpha_peak']].mean() * 100).round(1))